In [3]:
from pyspark.sql.functions import col , isnull ,when
from pyspark.sql.types import TimestampType


StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 5, Finished, Available, Finished)

In [8]:
# load thhe json data into a sark dataframe
df = spark.read.option("multiline","true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 10, Finished, Available, Finished)

In [17]:
df

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 19, Finished, Available, Finished)

DataFrame[id: string, longitude: double, latitude: double, elevation: double, title: string, place_description: string, sig: bigint, mag: double, magType: string, time: bigint, updated: bigint]

In [22]:
display(df)

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 24, Finished, Cancelled, Cancelled)

SynapseWidget(Synapse.DataFrame, e281fbf8-81fd-435e-a86c-2198bdd66005)

In [16]:
# Reshape earthquake data

df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )

)

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 18, Finished, Available, Finished)

In [19]:
# Validate date ; checking for missing or null values

df = (
    df
    .withColumn('longitude',when(isnull(col('longitude')),0).otherwise(col('longitude')))
    .withColumn('latitude',when(isnull(col('latitude')),0).otherwise(col('latitude')))
    .withColumn('time',when(isnull(col('time')),0).otherwise(col('time')))
)

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 21, Finished, Available, Finished)

In [21]:
# convert 'time' and ' udated' to timestamp

df = (
    df
    .withColumn('time',(col('time')/1000).cast(TimestampType()))
    .withColumn('updated',(col('updated')/1000).cast(TimestampType()))
)

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 23, Finished, Available, Finished)

In [23]:
# append to the Silver table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, 61a61443-2693-43ac-878a-b206e6bac882, 25, Finished, Available, Finished)